# AHR_RL P20: black-box SAC pre-trained on synthetic races, validated and tested on real ones

A synthetic market generator (adapted from `betfair-rl/tape_synth.py`) is fitted on the real **TRAIN** races only.
The agent pre-trains on a **fresh synthetic race every episode**, so it never sees the same race twice and has nothing to memorise.
It is then fine-tuned on real TRAIN races, its checkpoint is picked on real **VAL** races, and it is scored **once** on real **TEST** races.

What is different from `tape_synth`, so the agent cannot learn something only the generator does:
* **No bridge to the closing price** (that bridge plants a trend). Prices are a latent random walk of real moves, plus real-sized steaming/drifting spells and a short-lived deviation that reverts.
* **Trades are informed**: they lean towards where the price goes next, so resting orders suffer real adverse selection (P15).
* **BSP and the winner come from the final book** with real errors and calibration.
* **Calibrated to real TRAIN races**: volatility by time to the off, 60s variance ratio, trade-side information and 30s momentum.

The **parity check** compares real and synthetic races on what a trader could exploit:
* momentum/reversal at 30s and 120s, and the drift-fade (P16);
* the cost of crossing, and resting-order fill rate and adverse selection (P15).

The **agent** is the P17b bracket agent: a whole trade per action, reward at the close, gamma 1, 12-step returns.
To encourage exploration:
* a long warm-up (40% of pre-training) of random strategies plus drift-fade demos;
* a higher entropy target;
* an entropy floor, so it never stops exploring.

**Controls**:
* a real-only baseline trained the same way from scratch: does synthetic data help?
* the zero-shot pretrained agent;
* an artefact check: the pretrained agent on unseen synthetic races vs real VAL.

**Run order** (on its own Colab runtime: a GPU runtime, and as many CPUs as you can get):
1. Cells 1-2: setup.
2. **Stage A** (about 30-60 min): fit the generator and run the parity check. Read the parity table first.
3. **Stage B** (several hours): pretrain, fine-tune, baseline, test. Every stage resumes: a finished stage is skipped on a re-run, but a stage cut off mid-way starts again.
4. **Review**.

Outputs: `My Drive/betfair stream data/rl_runs_ahr_rl/synth_sac` (`parity/`, `pretrain/`, `finetune/`, `baseline/`, `results.csv`, `verdict.json`).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/betfair stream data'

In [ ]:
!git clone -b claude/wizardly-clarke-rb37hl https://github.com/tristanhowells/AHR_RL.git /content/AHR_RL || (cd /content/AHR_RL && git pull)
%cd /content/AHR_RL
!pip -q install orjson gymnasium>=1.0
!python -m pytest -q tests/test_proposals.py -k synth

## Stage A: fit the generator on TRAIN races, then parity (real TRAIN vs synthetic)

Look at the `critical` rows of the parity table:
* `PARITY OK`: synthetic races behave like real ones where a trader could exploit a difference.
* Any failed critical row is a difference the agent could learn. Stage B still runs, and two things protect against it: the fine-tune on real races with VAL selection, and the artefact check, which measures whether it happened.

To stop instead, add `--require-parity` to Stage B.

In [ ]:
import os
DRIVE = '/content/drive/MyDrive/betfair stream data'
TAPES = f'{DRIVE}/rl_tapes_v1'
NCPU = os.cpu_count()
if not os.path.isdir('/content/tapes') or not os.listdir('/content/tapes'):
    !mkdir -p /content/tapes && rsync -a "{TAPES}/" /content/tapes/
print(len(os.listdir('/content/tapes')), 'tapes')  # should be ~1650+: if far fewer, delete /content/tapes and re-run
P20 = f'{DRIVE}/rl_runs_ahr_rl/synth_sac'
WORK = '/content/p20_work'  # synthetic races are written here (local disk), never to Drive

!python -m ahr_rl.synth_sac --tapes "/content/tapes/*.npz" --out "{P20}" --work {WORK} --stages fit,parity --workers {NCPU}

## Stage B: pretrain on endless synthetic races, fine-tune on real TRAIN races, real-only baseline, TEST once

Defaults:
* **Pretrain**: 1.5M steps, of which 40% is warm-up. Alpha starts at 0.2 with a floor of 0.02; target entropy −0.25 × 6.
* **Fine-tune**: 300k steps at lr 1e-4, entropy floor 0.005.
* **Baseline**: 300k steps.

Append flags to the command to change these, for example `--pretrain-steps 3000000` or `--min-matched 20000`.
`--min-matched 20000` restricts TRAIN, VAL and TEST to the bigger races, but it leaves only about 13 TEST races, so it is not recommended.

Re-run this cell after a disconnect: finished stages are skipped.

In [ ]:
import os
DRIVE = '/content/drive/MyDrive/betfair stream data'
TAPES = f'{DRIVE}/rl_tapes_v1'
NCPU = os.cpu_count()
if not os.path.isdir('/content/tapes') or not os.listdir('/content/tapes'):
    !mkdir -p /content/tapes && rsync -a "{TAPES}/" /content/tapes/
print(len(os.listdir('/content/tapes')), 'tapes')  # should be ~1650+: if far fewer, delete /content/tapes and re-run
P20 = f'{DRIVE}/rl_runs_ahr_rl/synth_sac'
WORK = '/content/p20_work'  # synthetic races are written here (local disk), never to Drive

!python -m ahr_rl.synth_sac --tapes "/content/tapes/*.npz" --out "{P20}" --work {WORK} --n-envs {max(4, NCPU)} --workers {NCPU}

## Review

In [ ]:
import os, json
import pandas as pd
import matplotlib.pyplot as plt
DRIVE = '/content/drive/MyDrive/betfair stream data'
P20 = f'{DRIVE}/rl_runs_ahr_rl/synth_sac'
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 30)
f = f'{P20}/parity/parity.csv'
if os.path.exists(f):
    par = pd.read_csv(f)
    print('=== parity (critical rows) ==='); print(par[par.critical].round(4).to_string(index=False))
for name in ('results.csv', 'artefact_check.csv'):
    f = f'{P20}/{name}'
    if os.path.exists(f):
        print(f'\n=== {name} ==='); print(pd.read_csv(f).round(3).to_string(index=False))
f = f'{P20}/verdict.json'
if os.path.exists(f):
    print('\n=== verdict ==='); print(json.dumps(json.load(open(f)), indent=1))
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
for stage in ('pretrain', 'finetune', 'baseline'):
    f = f'{P20}/{stage}/train_log.csv'
    if os.path.exists(f):
        lg = pd.read_csv(f)
        ax[0].plot(lg.steps, lg.val_green, marker='o', label=stage)
        if 'alpha' in lg:
            ax[1].plot(lg.steps, lg.alpha, marker='.', label=stage)
ax[0].axhline(0, color='grey', lw=0.8); ax[0].set_title('real VAL green $/race by checkpoint'); ax[0].legend()
ax[1].set_title('entropy weight alpha'); ax[1].legend(); plt.show()